# Financial Risk Clustering

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Configuration

In [2]:
DATASET_PATH = 'archive/Financial Risk Classification Dataset.csv'
LABELED_OUTPUT_PATH = 'archive/financial_risk_labeled.csv'

FEATURES_TO_KEEP = [
    'loan_to_income_ratio', 'expenses_to_income_ratio', 'savings_to_income_ratio',
    'debt_to_income_ratio', 'previous_default_count',
    'loan_duration_months', 'interest_rate', 'age', 'employment_stability_years'
]

PROFILE_NAMES = {
    0: 'Financially Stable',
    1: 'Moderate Financial Capacity',
    2: 'Financially Vulnerable'
}

PROFILE_NOTE = (
    'Cluster merupakan hasil segmentasi tanpa label (unsupervised learning) '
    'dan tidak merepresentasikan risiko gagal bayar aktual maupun keputusan kredit resmi.'
)

## 2. Feature Engineering

In [3]:
def prepare_features(df):
    df = df.copy()
    df['annual_income'] = df['annual_income'].replace(0, 1)

    df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income']
    df['expenses_to_income_ratio'] = df['monthly_expenses'] / (df['annual_income'] / 12)
    df['savings_to_income_ratio'] = df['savings_balance'] / df['annual_income']

    missing_columns = [column for column in FEATURES_TO_KEEP if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    if 'credit_score' in df.columns:
        df = df.drop(columns=['credit_score'])

    X = df[FEATURES_TO_KEEP].values
    return df, X

## 3. K-Means Label Generation

In [4]:
def find_optimal_k(X, min_k=3, max_k=5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    best_k = min_k
    best_score = -1
    silhouette_scores = {}

    print('Evaluating Silhouette Scores:')
    for k in range(min_k, max_k + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        silhouette_scores[k] = score
        print(f' - K={k}: Silhouette Score = {score:.4f}')

        if score > best_score:
            best_score = score
            best_k = k

    print(f'-> Optimal K based on Silhouette Score: {best_k}')
    return best_k, X_scaled, silhouette_scores


def fit_kmeans(X_scaled, best_k):
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
    raw_cluster_labels = kmeans.fit_predict(X_scaled)
    return kmeans, raw_cluster_labels


def describe_cluster(row, global_means):
    characteristics = []

    if row['loan_to_income_ratio'] < global_means['loan_to_income_ratio']:
        characteristics.append('rasio pinjaman terhadap pendapatan relatif lebih rendah')
    else:
        characteristics.append('rasio pinjaman terhadap pendapatan relatif lebih tinggi')

    if row['expenses_to_income_ratio'] < global_means['expenses_to_income_ratio']:
        characteristics.append('beban pengeluaran terhadap pendapatan relatif lebih rendah')
    else:
        characteristics.append('beban pengeluaran terhadap pendapatan relatif lebih tinggi')

    if row['savings_to_income_ratio'] > global_means['savings_to_income_ratio']:
        characteristics.append('cadangan tabungan relatif lebih tinggi')
    else:
        characteristics.append('cadangan tabungan relatif lebih rendah')

    if row['employment_stability_years'] > global_means['employment_stability_years']:
        characteristics.append('stabilitas pekerjaan relatif lebih kuat')
    else:
        characteristics.append('stabilitas pekerjaan relatif lebih rendah')

    if row['interest_rate'] < global_means['interest_rate']:
        characteristics.append('eksposur bunga relatif lebih rendah')
    else:
        characteristics.append('eksposur bunga relatif lebih tinggi')

    return characteristics


def build_cluster_interpretation(kmeans, scaler, X):
    centroids = pd.DataFrame(
        scaler.inverse_transform(kmeans.cluster_centers_),
        columns=FEATURES_TO_KEEP
    )
    centroids.insert(0, 'raw_cluster_id', centroids.index)

    global_means = pd.Series(X.mean(axis=0), index=FEATURES_TO_KEEP)
    centers_scaled = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES_TO_KEEP)
    centroids['repayment_capacity_pressure'] = (
        0.4 * centers_scaled['loan_to_income_ratio']
        + 0.4 * centers_scaled['expenses_to_income_ratio']
        + 0.2 * centers_scaled['debt_to_income_ratio']
    )
    centroids['financial_resilience_score'] = (
        0.6 * centers_scaled['savings_to_income_ratio']
        + 0.4 * centers_scaled['employment_stability_years']
    )
    centroids['financial_behavior_pressure'] = centers_scaled[
        ['previous_default_count', 'interest_rate']
    ].mean(axis=1)
    centroids['profile_pressure_score'] = (
        centroids['repayment_capacity_pressure']
        + centroids['financial_behavior_pressure']
        - centroids['financial_resilience_score']
    )

    ordered_cluster_ids = centroids.sort_values('profile_pressure_score')['raw_cluster_id'].tolist()
    cluster_mapping = {
        raw_cluster_id: profile_label
        for profile_label, raw_cluster_id in enumerate(ordered_cluster_ids)
    }
    centroids['financial_profile_label'] = centroids['raw_cluster_id'].map(cluster_mapping)
    centroids['profile_name'] = centroids['financial_profile_label'].map(PROFILE_NAMES)

    print('\nCatatan interpretasi:')
    print(PROFILE_NOTE)

    print('\nRata-rata global fitur final:')
    print(global_means.round(4))

    print('\nAnalisis post-hoc cluster asli K-Means sebelum pelabelan final:')
    for _, row in centroids.sort_values('raw_cluster_id').iterrows():
        raw_cluster_id = int(row['raw_cluster_id'])
        final_label = int(row['financial_profile_label'])
        print(f"\nRaw Cluster {raw_cluster_id} -> {row['profile_name']} (label final {final_label})")
        print('Centroid:')
        centroid_values = pd.to_numeric(row[FEATURES_TO_KEEP])
        print(centroid_values.round(4))
        print('Selisih dari rata-rata global:')
        print((centroid_values - global_means).round(4))
        print('Karakteristik dominan:')
        for characteristic in describe_cluster(row, global_means):
            print(f'- {characteristic}')
        print('Ringkasan dimensi:')
        print(f"- Repayment Capacity pressure: {row['repayment_capacity_pressure']:.4f}")
        print(f"- Financial Resilience score: {row['financial_resilience_score']:.4f}")
        print(f"- Financial Behavior pressure: {row['financial_behavior_pressure']:.4f}")

    print('\nMapping hasil analisis ke label final:')
    for raw_cluster_id, final_label in sorted(cluster_mapping.items()):
        print(f"- Raw Cluster {raw_cluster_id} -> Label {final_label} ({PROFILE_NAMES[final_label]})")

    return centroids, global_means, cluster_mapping


def assign_profile_labels(raw_cluster_labels, cluster_mapping):
    return np.array([cluster_mapping[label] for label in raw_cluster_labels])

## 4. Execution

In [5]:
print('Loading data...')
df_raw = pd.read_csv(DATASET_PATH)

print('Feature Engineering...')
df_features, X = prepare_features(df_raw)

print('\nStarting K-Means Clustering to discover raw financial profile segments...')
optimal_k, X_scaled, silhouette_scores = find_optimal_k(X)
kmeans_model, raw_cluster_labels = fit_kmeans(X_scaled, optimal_k)
cluster_analysis, global_means, cluster_mapping = build_cluster_interpretation(
    kmeans_model,
    StandardScaler().fit(X),
    X
)
profile_labels = assign_profile_labels(raw_cluster_labels, cluster_mapping)

df_labeled = df_features.copy()
df_labeled['raw_cluster_id'] = raw_cluster_labels
df_labeled['financial_profile_label'] = profile_labels

print('\nDistribusi raw cluster K-Means:')
print(pd.Series(raw_cluster_labels).value_counts().sort_index())

print('\nDistribusi label profil finansial setelah interpretasi:')
print(pd.Series(profile_labels).value_counts().sort_index())

df_labeled.to_csv(LABELED_OUTPUT_PATH, index=False)
print(f'\nLabeled data saved to: {LABELED_OUTPUT_PATH}')

Loading data...
Feature Engineering...

Starting K-Means Clustering to discover raw financial profile segments...
Evaluating Silhouette Scores:


C:\Users\rhzain\miniconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:110: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\rhzain\miniconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True, text=True)
  File "C:\Users\rhzain\miniconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rhzain\miniconda3\Lib\subprocess.py", line 1036, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^

 - K=3: Silhouette Score = 0.0884


 - K=4: Silhouette Score = 0.0787


 - K=5: Silhouette Score = 0.0872
-> Optimal K based on Silhouette Score: 3

Catatan interpretasi:
Cluster merupakan hasil segmentasi tanpa label (unsupervised learning) dan tidak merepresentasikan risiko gagal bayar aktual maupun keputusan kredit resmi.

Rata-rata global fitur final:
loan_to_income_ratio           0.4501
expenses_to_income_ratio       0.5366
savings_to_income_ratio        0.2747
debt_to_income_ratio           0.4748
previous_default_count         0.4840
loan_duration_months          36.0768
interest_rate                 10.4989
age                           41.0138
employment_stability_years    14.7094
dtype: float64

Analisis post-hoc cluster asli K-Means sebelum pelabelan final:

Raw Cluster 0 -> Moderate Financial Capacity (label final 1)
Centroid:
loan_to_income_ratio           0.7654
expenses_to_income_ratio       0.8634
savings_to_income_ratio        0.4834
debt_to_income_ratio           0.4909
previous_default_count         0.5082
loan_duration_months          